In [0]:
# Configuracion
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "banco"
SCHEMA = "banking_analytics"

GOLD_TABLES = {
    "dim_customer": f"{CATALOG}.{SCHEMA}.dim_customer",
    "dim_account": f"{CATALOG}.{SCHEMA}.dim_account",
    "dim_branch": f"{CATALOG}.{SCHEMA}.dim_branch",
    "dim_date": f"{CATALOG}.{SCHEMA}.dim_date",
    "fact_transactions": f"{CATALOG}.{SCHEMA}.fact_transactions"
}

In [0]:
# Carga de tablas gold
df_fact = spark.table(
    GOLD_TABLES["fact_transactions"]
)

df_customer = spark.table(
    GOLD_TABLES["dim_customer"]
)

df_account = spark.table(
    GOLD_TABLES["dim_account"]
)

df_branch = spark.table(
    GOLD_TABLES["dim_branch"]
)

df_date = spark.table(
    GOLD_TABLES["dim_date"]
)

print("Tablas Gold cargadas correctamente.")

In [0]:
# Validamos datos de entrada
print("FACT TRANSACTIONS:", df_fact.count())
print("DIM CUSTOMER:", df_customer.count())
print("DIM ACCOUNT:", df_account.count())
print("DIM BRANCH:", df_branch.count())
print("DIM DATE:", df_date.count())

In [0]:
# Creación de Kpis generales
df_kpi_general = df_fact.agg(
    F.sum("transaction_count").alias("total_transactions"),
    F.countDistinct("customer_key").alias("active_customers"),
    F.countDistinct("account_key").alias("active_accounts"),
    F.countDistinct("branch_key").alias("active_branches")
)

display(df_kpi_general)

In [0]:
# Indicadores por moneda
df_kpi_currency = (
    df_fact
    .groupBy("currency")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.min("amount").alias("minimum_amount"),
        F.max("amount").alias("maximum_amount"),
        F.countDistinct("customer_key").alias("active_customers")
    )
    .orderBy("currency")
)

display(df_kpi_currency)

In [0]:
# Identificar estados disponibles
df_fact.select("status") \
    .distinct() \
    .orderBy("status") \
    .show(truncate=False)

In [0]:
# Analizar transacciones por estado y memoria
df_kpi_status = (
    df_fact
    .groupBy("currency", "status")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers")
    )
    .orderBy("currency", "status")
)

display(df_kpi_status)

In [0]:
# KPIs de operaciones completadas

df_kpi_completed = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("currency")
    .agg(
        F.sum("transaction_count").alias("completed_transactions"),
        F.sum("amount").alias("completed_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
    .orderBy("currency")
)

display(df_kpi_completed)


In [0]:
# Indicadores de efectividad y rechazo

df_kpi_effectiveness = (
    df_fact
    .groupBy("currency")
    .agg(
        F.count("*").alias("total_transactions"),

        F.sum(
            F.when(F.col("status") == "COMPLETED", 1).otherwise(0)
        ).alias("completed_transactions"),

        F.sum(
            F.when(F.col("status") == "REJECTED", 1).otherwise(0)
        ).alias("rejected_transactions"),

        F.sum(
            F.when(F.col("status") == "PENDING", 1).otherwise(0)
        ).alias("pending_transactions")
    )
    .withColumn(
        "completion_rate",
        F.round(
            F.col("completed_transactions") /
            F.col("total_transactions") * 100, 2
        )
    )
    .withColumn(
        "rejection_rate",
        F.round(
            F.col("rejected_transactions") /
            F.col("total_transactions") * 100, 2
        )
    )
    .withColumn(
        "pending_rate",
        F.round(
            F.col("pending_transactions") /
            F.col("total_transactions") * 100, 2
        )
    )
    .orderBy("currency")
)

display(df_kpi_effectiveness)


In [0]:
# Analisis mensual
df_monthly_transactions = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .join(
        df_date,
        df_fact["date_key"] == df_date["date_key"],
        "inner"
    )
    .groupBy(
        df_date["year"],
        df_date["month"],
        df_date["month_name"],
        df_fact["currency"]
    )
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers")
    )
    .orderBy(
        "year",
        "month",
        "currency"
    )
)

display(df_monthly_transactions)


In [0]:
# Analisis de crecimiento mensual

# Definimos una ventana independiente para cada moneda
window_monthly = (
    Window
    .partitionBy("currency")
    .orderBy("year", "month")
)

df_monthly_growth = (
    df_monthly_transactions

    # Recuperamos los valores del mes anterior
    .withColumn(
        "previous_month_transactions",
        F.lag("total_transactions").over(window_monthly)
    )
    .withColumn(
        "previous_month_amount",
        F.lag("total_amount").over(window_monthly)
    )

    # Calculamos la variación porcentual de transacciones
    .withColumn(
        "transactions_growth_pct",
        F.when(
            F.col("previous_month_transactions").isNotNull()
            & (F.col("previous_month_transactions") != 0),
            F.round(
                (
                    F.col("total_transactions")
                    - F.col("previous_month_transactions")
                ) / F.col("previous_month_transactions") * 100,
                2
            )
        )
    )

    # Calculamos la variación porcentual del importe
    .withColumn(
        "amount_growth_pct",
        F.when(
            F.col("previous_month_amount").isNotNull()
            & (F.col("previous_month_amount") != 0),
            F.round(
                (
                    F.col("total_amount")
                    - F.col("previous_month_amount")
                ) / F.col("previous_month_amount") * 100,
                2
            )
        )
    )

    .select(
        "year",
        "month",
        "month_name",
        "currency",
        "total_transactions",
        "previous_month_transactions",
        "transactions_growth_pct",
        "total_amount",
        "previous_month_amount",
        "amount_growth_pct"
    )
    .orderBy("currency", "year", "month")
)

display(df_monthly_growth)


In [0]:

# Tipos de transaccion

df_fact.select("transaction_type") \
    .distinct() \
    .orderBy("transaction_type") \
    .show(truncate=False)


In [0]:

# Canales disponibles
df_fact.select("channel") \
    .distinct() \
    .orderBy("channel") \
    .show(truncate=False)


In [0]:
# Analisis por tipo de transacción
df_kpi_transaction_type = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("transaction_type", "currency")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts")
    )
    .orderBy("transaction_type", "currency")
)

display(df_kpi_transaction_type)


In [0]:
# Analisis por canal bancario

df_kpi_channel = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("channel", "currency")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts")
    )
    .orderBy("channel", "currency")
)

display(df_kpi_channel)


In [0]:

# Consultas
df_kpi_type_channel = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy(
        "transaction_type",
        "channel",
        "currency"
    )
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers")
    )
    .orderBy(
        "transaction_type",
        "currency",
        F.desc("total_transactions")
    )
)

display(df_kpi_type_channel)


In [0]:
# Participación Porcentual por canal
window_channel = Window.partitionBy(
    "transaction_type",
    "currency"
)

df_channel_share = (
    df_kpi_type_channel
    .withColumn(
        "total_type_currency",
        F.sum("total_transactions").over(window_channel)
    )
    .withColumn(
        "channel_share_pct",
        F.round(
            F.col("total_transactions") /
            F.col("total_type_currency") * 100,
            2
        )
    )
    .select(
        "transaction_type",
        "currency",
        "channel",
        "total_transactions",
        "total_type_currency",
        "channel_share_pct"
    )
    .orderBy(
        "transaction_type",
        "currency",
        F.desc("channel_share_pct")
    )
)

display(df_channel_share)

In [0]:
# Clientes únicos que utilizaron cada canal
df_customers_by_channel = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("currency", "channel")
    .agg(
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.sum("transaction_count").alias("total_transactions")
    )
)

# Total de clientes únicos activos por moneda
df_total_customers_currency = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("currency")
    .agg(
        F.countDistinct("customer_key").alias("total_active_customers")
    )
)

# Calcular penetración de clientes por canal
df_channel_penetration = (
    df_customers_by_channel
    .join(
        df_total_customers_currency,
        on="currency",
        how="inner"
    )
    .withColumn(
        "customer_penetration_pct",
        F.round(
            F.col("active_customers") /
            F.col("total_active_customers") * 100,
            2
        )
    )
    .orderBy("currency", F.desc("customer_penetration_pct"))
)

display(df_channel_penetration)

In [0]:
# Identificar cuántos canales utiliza cada cliente por moneda
df_customer_channels = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("customer_key", "currency")
    .agg(
        F.countDistinct("channel").alias("channels_used"),
        F.sum("transaction_count").alias("total_transactions")
    )
)

# Distribución de clientes según cantidad de canales utilizados
df_multichannel = (
    df_customer_channels
    .groupBy("currency", "channels_used")
    .agg(
        F.count("*").alias("total_customers"),
        F.avg("total_transactions").alias("avg_transactions")
    )
    .withColumn(
        "customer_share_pct",
        F.round(
            F.col("total_customers") /
            F.sum("total_customers").over(
                Window.partitionBy("currency")
            ) * 100,
            2
        )
    )
    .orderBy("currency", "channels_used")
)

display(df_multichannel)

In [0]:
# Segmento con mejores opciones
df_customer_segment_volume = (
    df_customer_channels
    .groupBy("currency", "channels_used")
    .agg(
        F.countDistinct("customer_key").alias("total_customers"),
        F.sum("total_transactions").alias("total_transactions"),
        F.avg("total_transactions").alias("avg_transactions_per_customer")
    )
)

window_segment = Window.partitionBy("currency")

df_customer_segment_volume = (
    df_customer_segment_volume
    .withColumn(
        "total_currency_transactions",
        F.sum("total_transactions").over(window_segment)
    )
    .withColumn(
        "transaction_share_pct",
        F.round(
            F.col("total_transactions") /
            F.col("total_currency_transactions") * 100,
            2
        )
    )
    .orderBy("currency", "channels_used")
)

display(df_customer_segment_volume)

In [0]:
df_segment_intensity = (
    df_customer_segment_volume
    .join(
        df_multichannel.select(
            "currency",
            "channels_used",
            "customer_share_pct"
        ),
        on=["currency", "channels_used"],
        how="inner"
    )
    .withColumn(
        "transaction_intensity_index",
        F.round(
            F.col("transaction_share_pct") /
            F.col("customer_share_pct") * 100,
            2
        )
    )
    .select(
        "currency",
        "channels_used",
        "customer_share_pct",
        "transaction_share_pct",
        "avg_transactions_per_customer",
        "transaction_intensity_index"
    )
    .orderBy("currency", "channels_used")
)

display(df_segment_intensity)

In [0]:
df_customer_channel_combinations = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("customer_key", "currency")
    .agg(
        F.array_sort(
            F.collect_set("channel")
        ).alias("channels"),
        F.sum("transaction_count").alias("total_transactions")
    )
    .withColumn(
        "channel_combination",
        F.concat_ws(" + ", F.col("channels"))
    )
)

df_combination_summary = (
    df_customer_channel_combinations
    .groupBy("currency", "channel_combination")
    .agg(
        F.countDistinct("customer_key").alias("total_customers"),
        F.sum("total_transactions").alias("total_transactions"),
        F.avg("total_transactions").alias("avg_transactions_per_customer")
    )
    .withColumn(
        "customer_share_pct",
        F.round(
            F.col("total_customers") /
            F.sum("total_customers").over(
                Window.partitionBy("currency")
            ) * 100,
            2
        )
    )
    .orderBy(
        "currency",
        F.desc("total_customers")
    )
)

display(df_combination_summary)

In [0]:
window_top_combinations = (
    Window
    .partitionBy("currency")
    .orderBy(F.desc("total_transactions"))
)

df_top_combinations = (
    df_combination_summary
    .withColumn(
        "ranking",
        F.row_number().over(window_top_combinations)
    )
    .filter(F.col("ranking") <= 10)
    .select(
        "currency",
        "ranking",
        "channel_combination",
        "total_customers",
        "total_transactions",
        "avg_transactions_per_customer",
        "customer_share_pct"
    )
    .orderBy("currency", "ranking")
)

display(df_top_combinations)

In [0]:
# Tablas gold de Kpis
# Nombre de la tabla Gold
TABLE_MONTHLY = f"{CATALOG}.{SCHEMA}.agg_monthly_transactions"

# Preparar DataFrame final
df_agg_monthly_transactions = (
    df_monthly_transactions
    .withColumn(
        "year_month",
        F.concat(
            F.col("year").cast("string"),
            F.lit("-"),
            F.lpad(F.col("month").cast("string"), 2, "0")
        )
    )
    .select(
        "year",
        "month",
        "year_month",
        "month_name",
        "currency",
        "total_transactions",
        "total_amount",
        "average_amount",
        "active_customers"
    )
)

display(df_agg_monthly_transactions)

In [0]:
# Guardar la tabla en delta
df_agg_monthly_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_MONTHLY)

In [0]:
# Validar persistencia
df_check = spark.table(TABLE_MONTHLY)

print(f"Filas persistidas: {df_check.count()}")

df_check.printSchema()

display(
    df_check.orderBy("year", "month", "currency")
)

In [0]:
TABLE_STATUS = f"{CATALOG}.{SCHEMA}.agg_transaction_status"

df_agg_transaction_status = (
    df_fact
    .groupBy("currency", "status")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
)

# Calcular la participación de cada estado dentro de su moneda
window_currency = Window.partitionBy("currency")

df_agg_transaction_status = (
    df_agg_transaction_status
    .withColumn(
        "currency_total_transactions",
        F.sum("total_transactions").over(window_currency)
    )
    .withColumn(
        "status_share_pct",
        F.round(
            F.col("total_transactions") /
            F.col("currency_total_transactions") * 100,
            2
        )
    )
    .orderBy("currency", "status")
)

display(df_agg_transaction_status)

In [0]:
# Crear la tabla delta del KPi
df_agg_transaction_status.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_STATUS)

In [0]:
# Validar persistencia
df_check_status = spark.table(TABLE_STATUS)

print(f"Filas persistidas: {df_check_status.count()}")

display(
    df_check_status.orderBy("currency", "status")
)

In [0]:
# Tipo de transaccion
TABLE_TYPE = f"{CATALOG}.{SCHEMA}.agg_transaction_type"

df_agg_transaction_type = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("transaction_type", "currency")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.min("amount").alias("minimum_amount"),
        F.max("amount").alias("maximum_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts")
    )
    .orderBy("transaction_type", "currency")
)

display(df_agg_transaction_type)

In [0]:
df_agg_transaction_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_TYPE)

In [0]:
# Validar persistencia
df_check_type = spark.table(TABLE_TYPE)

print(f"Filas persistidas: {df_check_type.count()}")

display(
    df_check_type.orderBy("transaction_type", "currency")
)

In [0]:
TABLE_CHANNEL = f"{CATALOG}.{SCHEMA}.agg_channel_performance"

df_agg_channel_performance = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("channel", "currency")
    .agg(
        F.sum("transaction_count").alias("total_transactions"),
        F.sum("amount").alias("total_amount"),
        F.avg("amount").alias("average_amount"),
        F.countDistinct("customer_key").alias("active_customers"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
    .orderBy("channel", "currency")
)

display(df_agg_channel_performance)

In [0]:
# Guardar en delta lake
df_agg_channel_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_CHANNEL)

In [0]:
df_check_channel = spark.table(TABLE_CHANNEL)

print(f"Filas persistidas: {df_check_channel.count()}")

display(
    df_check_channel.orderBy("channel", "currency")
)

In [0]:
# agrgacion customer
TABLE_MULTICHANNEL = f"{CATALOG}.{SCHEMA}.agg_customer_multichannel"

df_agg_customer_multichannel = (
    df_fact
    .filter(F.col("status") == "COMPLETED")
    .groupBy("customer_key", "currency")
    .agg(
        F.countDistinct("channel").alias("channels_used"),
        F.sum("transaction_count").alias("total_transactions"),
        F.countDistinct("account_key").alias("active_accounts"),
        F.countDistinct("branch_key").alias("active_branches")
    )
)

display(
    df_agg_customer_multichannel
    .orderBy("currency", "channels_used")
)

In [0]:
# Validar resultados 
df_agg_customer_multichannel.groupBy("currency").agg(
    F.count("*").alias("total_customers"),
    F.min("channels_used").alias("min_channels"),
    F.max("channels_used").alias("max_channels"),
    F.round(F.avg("channels_used"), 2).alias("avg_channels")
).orderBy("currency").show()

In [0]:
df_agg_customer_multichannel.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_MULTICHANNEL)

In [0]:
# Validar persistencia
df_check_multichannel = spark.table(TABLE_MULTICHANNEL)

print(f"Filas persistidas: {df_check_multichannel.count()}")

display(
    df_check_multichannel
    .orderBy("currency", "channels_used")
)

In [0]:
AGG_TABLES = {
    "agg_monthly_transactions": 44,
    "agg_transaction_status": 6,
    "agg_transaction_type": 8,
    "agg_channel_performance": 10,
    "agg_customer_multichannel": 14332
}

audit_results = []

for table_name, expected_count in AGG_TABLES.items():

    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"

    actual_count = spark.table(full_table_name).count()

    audit_results.append((
        table_name,
        expected_count,
        actual_count,
        "PASS" if actual_count == expected_count else "FAIL"
    ))

df_gold_audit = spark.createDataFrame(
    audit_results,
    ["table_name", "expected_count", "actual_count", "validation_status"]
)

display(df_gold_audit)